# Tutorial 2 — Safe Data Loading

**Rule:** Never trust the file. Always inspect the DataFrame.

In this notebook you will practice:
- Loading CSVs safely
- Inspecting shape, columns, dtypes, and missing values
- Forcing correct dtypes
- Handling missing values thoughtfully
- Setting a meaningful index

---
## Part A — Load & Inspect

**Step 1:** Load the CSV file `data.csv` into a DataFrame named `df`.

In [ ]:
import pandas as pd

df = pd.read_csv("data.csv")

**Step 2:** Print the shape, first 5 rows, last 5 rows, and column names.

In [ ]:
print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nLast 5 rows:")
print(df.tail())
print("\nColumn names:")
print(df.columns.tolist())

### ✅ Part A — Assertions
Run the cell below to verify your work.

In [ ]:
# --- Part A Asserts ---
assert isinstance(df, pd.DataFrame), "df must be a DataFrame"
assert df.shape == (5, 5), f"Expected shape (5, 5), got {df.shape}"
expected_cols = ["ID", "Math", "Physics", "CS", "Class"]
assert df.columns.tolist() == expected_cols, f"Columns mismatch: {df.columns.tolist()}"
assert df.iloc[0, 0] == 1, "First row ID should be 1"
assert df.iloc[4, 0] == 5, "Last row ID should be 5"
print("✅ Part A passed!")

---
## Part B — Missing Values

**Step 1:** Use `.isna()` and `.sum()` to count missing values per column.

Store the result in a variable named `missing_counts`.

In [ ]:
missing_counts = df.isna().sum()
print(missing_counts)

**Step 2 (Discussion — no assert):** Which columns are dangerous for ML if they have missing values?

Write your answer as a comment below.

In [ ]:
# Answer:
# Math, Physics, and CS are dangerous because ML models need numeric inputs.
# ID is just an identifier.
# Class is the target label — missing there would be critical too.

### ✅ Part B — Assertions

In [ ]:
# --- Part B Asserts ---
assert missing_counts["Math"] == 1, f"Math should have 1 missing, got {missing_counts['Math']}"
assert missing_counts["Physics"] == 1, f"Physics should have 1 missing, got {missing_counts['Physics']}"
assert missing_counts["CS"] == 1, f"CS should have 1 missing, got {missing_counts['CS']}"
assert missing_counts["ID"] == 0, f"ID should have 0 missing, got {missing_counts['ID']}"
assert missing_counts["Class"] == 0, f"Class should have 0 missing, got {missing_counts['Class']}"
print("✅ Part B passed!")

---
## Part C — dtypes

**Step 1:** Print the dtypes of `df`.

In [ ]:
print(df.dtypes)

Notice that `Math`, `Physics`, and `CS` are `float64` because they contain NaN.

**Step 2:** Load the CSV again — this time force `ID` to be `int` and `Class` to be `int`.

Store the result in a variable named `df_typed`.

In [ ]:
df_typed = pd.read_csv(
    "data.csv",
    dtype={
        "ID": int,
        "Class": int
    }
)
print(df_typed.dtypes)

**Step 3 (Discussion — no assert):** Explain why forcing dtypes matters for ML.

Write your answer as a comment below.

In [ ]:
# Answer:
# ML models require numeric inputs. If a column is 'object' (strings/mixed),
# it cannot be used directly and will cause errors. Forcing dtypes ensures
# the data is in the expected numeric format, preventing silent bugs.

### ✅ Part C — Assertions

In [ ]:
# --- Part C Asserts ---
assert df_typed["ID"].dtype == int or df_typed["ID"].dtype == "int64", f"ID dtype should be int, got {df_typed['ID'].dtype}"
assert df_typed["Class"].dtype == int or df_typed["Class"].dtype == "int64", f"Class dtype should be int, got {df_typed['Class'].dtype}"
assert df_typed["Math"].dtype == float or df_typed["Math"].dtype == "float64", f"Math should remain float64 due to NaN, got {df_typed['Math'].dtype}"
print("✅ Part C passed!")

---
## Part D — Missing Value Handling

**Step 1:** Create a copy of `df` called `df_zero` and use `fillna(0)` to fill missing values.

In [ ]:
df_zero = df.copy()
df_zero = df_zero.fillna(0)
print(df_zero)

**Step 2:** Now impute missing values using the **mean** of each column.

Store the result in `df_mean`.

In [ ]:
df_mean = df.copy()
for col in ["Math", "Physics", "CS"]:
    df_mean[col] = df_mean[col].fillna(df_mean[col].mean())

print(df_mean)

**Step 3 (Discussion — no assert):** Why might `fillna(0)` destroy signal?

Write your answer as a comment below.

In [ ]:
# Answer:
# fillna(0) assumes the missing value is zero, which is often wrong.
# For example, a missing test score replaced with 0 makes the model think
# the student scored 0, distorting patterns and biasing predictions.
# A mean/median imputation preserves the overall distribution better.

### ✅ Part D — Assertions

In [ ]:
# --- Part D Asserts ---
assert df_zero.isna().sum().sum() == 0, "df_zero should have no missing values"
assert df_mean.isna().sum().sum() == 0, "df_mean should have no missing values"
# Check that mean-imputed values are not zero
assert df_mean.loc[0, "Physics"] != 0, "Mean-imputed Physics should not be 0"
assert df_mean.loc[2, "Math"] != 0, "Mean-imputed Math should not be 0"
assert df_mean.loc[3, "CS"] != 0, "Mean-imputed CS should not be 0"
print("✅ Part D passed!")

---
## Part E — Index

**Step 1:** Load `data.csv` again but this time set `ID` as the index column.

Store the result in `df_idx`.

In [ ]:
df_idx = pd.read_csv("data.csv", index_col="ID")
print(df_idx)

**Step 2:** Access the row with ID=3 using `.loc` and store it in `row_3`.

In [ ]:
row_3 = df_idx.loc[3]
print(row_3)

### ✅ Part E — Assertions

In [ ]:
# --- Part E Asserts ---
assert df_idx.index.name == "ID", f"Index name should be 'ID', got {df_idx.index.name}"
assert 3 in df_idx.index, "ID=3 should be in the index"
assert row_3["Math"] != row_3["Math"] or pd.isna(row_3["Math"]), "Row for ID=3 should have NaN in Math"
assert row_3["Class"] == 1, f"Row for ID=3 should have Class=1, got {row_3['Class']}"
print("✅ Part E passed!")

---
## Summary

After this tutorial you should understand:
- ✅ `read_csv` is powerful but can silently break data
- ✅ Always inspect after loading: `.head()`, `.dtypes`, `.isna().sum()`
- ✅ NaN in numeric columns → `float64`
- ✅ Mixed types → `object`
- ✅ `.fillna(0)` may destroy the signal
- ✅ Index should represent identity
- ✅ Most ML bugs start here if ignored